In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pacer_feedback_exp.preference import make_synthetic_true_preference

# ----------------------------
# Load data and saved results
# ----------------------------

DATA_PATH = "data/train_with_aspect_scores_sentences.csv"
CSV_PATH = "outputs/convergence_seeds_static/convergence_all_runs.csv"
# FIG_DIR = "figures_static"

# os.makedirs(FIG_DIR, exist_ok=True)

data = pd.read_csv(DATA_PATH)
df_all = pd.read_csv(CSV_PATH)

phi_cols = [c for c in data.columns if c.startswith("asp_")]
user_id = "AG73BVBKUOH22USSFJA5ZWL7AKXA"

# ----------------------------
# Reconstruct w_true per seed
# ----------------------------

seed_refs = []

for seed in sorted(df_all["seed"].unique()):
    w_true = make_synthetic_true_preference(
        df_sent=data,
        user_id=user_id,
        phi_cols=phi_cols,
        beta=8.0,
        noise_scale=0.03,
        seed=int(seed),
    )

    seed_refs.append({
        "seed": int(seed),
        "A_pref_self": float(np.dot(w_true, w_true)),
        "A_simplex_upper": float(np.max(w_true)),
    })

df_refs = pd.DataFrame(seed_refs)

print(df_refs)
print("Mean exact-recovery preference alignment:", df_refs["A_pref_self"].mean())
print("Mean simplex upper bound:", df_refs["A_simplex_upper"].mean())

# Merge references into saved experiment results
df_all = df_all.merge(df_refs, on="seed", how="left")
df_all.head()

   seed  A_pref_self  A_simplex_upper
0     0     0.215505         0.418156
1     1     0.204676         0.401911
2     2     0.206856         0.404963
3     3     0.198568         0.392937
4     4     0.206471         0.405202
5     5     0.205162         0.402090
6     6     0.209517         0.409410
7     7     0.210330         0.410603
8     8     0.206063         0.403300
9     9     0.211801         0.412521
Mean exact-recovery preference alignment: 0.20749503523111343
Mean simplex upper bound: 0.40610910654067994


,seed,variant,extractor,policy,t,feedback,A_pref,A_pref_avg,A_evid,cos_omd,cos_omd_avg,kl_omd_to_true,A_pref_self,A_simplex_upper
0,0,Static-MMR,mmr,static,0,0.570732,0.1,0.1,0.123737,0.681194,0.681194,0.281017,0.215505,0.418156
1,0,Static-MMR,mmr,static,1,0.570732,0.1,0.1,0.123737,0.681194,0.681194,0.281017,0.215505,0.418156
2,0,Static-MMR,mmr,static,2,0.570732,0.1,0.1,0.123737,0.681194,0.681194,0.281017,0.215505,0.418156
3,0,Static-MMR,mmr,static,3,0.570732,0.1,0.1,0.123737,0.681194,0.681194,0.281017,0.215505,0.418156
4,0,Static-MMR,mmr,static,4,0.570732,0.1,0.1,0.123737,0.681194,0.681194,0.281017,0.215505,0.418156


In [2]:
df_all.to_csv("outputs/convergence_seeds_final/convergence_all_runs_with_refs.csv", index=False)